In [0]:
# Set spark config

spark.conf.set("spark.sql.shuffle.partitions", 50)

In [0]:
# Read data from Volumes path

df = spark.read.format("csv").option("header", True).load("/Volumes/dev-external-catalog/default/test-volume/Employee_Attrition.csv")
display(df)

In [0]:
# Transformations on dataframe and write data into delta table

from pyspark.sql.functions import col

high_risk_df = df.filter(
    (col("Attrition") == "No") & (col("JobSatisfaction").cast("int") < 3)
)

selected_columns = [
    "EmployeeNumber", "EmployeeName", "Department", "JobRole", "JobSatisfaction",
    "Age", "Gender", "MaritalStatus", "MonthlyIncome", "OverTime", "YearsAtCompany"
]
high_risk_selected_df = high_risk_df.select(*[c for c in selected_columns if c in high_risk_df.columns])

high_risk_selected_df.write.format("delta").mode("overwrite").saveAsTable("`dev-external-catalog`.default.high_risk_attrition_employees")

In [0]:
# List down delta table versions

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`dev-external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()
display(history_df.select("version", "timestamp", "operation"))

In [0]:
# Read delta table into a Spark DataFrame

delta_df = spark.read.format("delta").table("`dev-external-catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
from pyspark.sql import Row

dummy_record = Row(
    EmployeeNumber="99999",
    Department="DummyDept",
    JobRole="DummyRole",
    JobSatisfaction="1",
    Age="30",
    Gender="Other",
    MaritalStatus="Single",
    MonthlyIncome="0",
    OverTime="No",
    YearsAtCompany="0"
)

dummy_df = spark.createDataFrame([dummy_record])
dummy_df.write.format("delta").mode("append").saveAsTable("`dev-external-catalog`.default.high_risk_attrition_employees")

In [0]:
# List down delta table versions

from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "`dev-external-catalog`.default.high_risk_attrition_employees")
history_df = delta_table.history()
display(history_df.select("version", "timestamp", "operation"))

In [0]:
delta_df = spark.read.format("delta").table("`dev-external-catalog`.default.high_risk_attrition_employees")
display(delta_df)

In [0]:
# Read delta table from specific version

version_number = 0  # Replace with desired version
delta_version_df = spark.read.option("versionAsOf", version_number).table("`dev-external-catalog`.default.high_risk_attrition_employees")
display(delta_version_df)

In [0]:
# Read delta table from specific timestamp

timestamp_str = "2026-03-11T15:35:27.401+00:00"  # Replace with desired timestamp
delta_timestamp_df = spark.read.option("timestampAsOf", timestamp_str).table("`dev-external-catalog`.default.high_risk_attrition_employees")
display(delta_timestamp_df)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS `dev-external-catalog`.default.employee_transformed_data
COMMENT 'Managed volume for storing data files'
""")

In [0]:
# Write partitioned data into Volume path

from pyspark.sql.functions import col, upper

# Example logical transformation: uppercase JobRole for all employees
transformed_df = df.withColumn("JobRole", upper(col("JobRole")))

output_path = "/Volumes/dev-external-catalog/default/employee_transformed_data/"

transformed_df.write.mode("overwrite").partitionBy("Department").format("parquet").save(output_path)
display(transformed_df)

In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS `dev-external-catalog`.default.employee_transformed_analysis_data
COMMENT 'Managed volume for storing data files'
""")

In [0]:
# Filter employees with MonthlyIncome greater than 5000 and write partitioned by Department

from pyspark.sql.functions import col

transformed_df = df.filter(col("MonthlyIncome").cast("int") > 5000)
transformed_df.write.partitionBy("Department").mode("overwrite").format("parquet").save("/Volumes/dev-external-catalog/default/employee_transformed_analysis_data/")
display(transformed_df)